In [107]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.model_selection import train_test_split
from trl import SFTTrainer
import torch
import pandas as pd

<h3 style="text-align:center"> downloading dataset</h3>

In [108]:
# Load the dataset
dataset = load_dataset("socratesft/SocSci210") 

# Load a mapping file
participant_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/participant_mapping.json",
    repo_type="dataset"
)

condition_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/condition_mapping.json",
    repo_type="dataset"
)

task_mapping = hf_hub_download(
    repo_id="socratesft/SocSci210",
    filename="metadata/task_mapping.json",
    repo_type="dataset"
)

print(condition_mapping)

Repo card metadata block was not found. Setting CardData to empty.


Resolving data files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/17 [00:00<?, ?it/s]

/home/jambo/.cache/huggingface/hub/datasets--socratesft--SocSci210/snapshots/048481111a4425ed83dc0eacf15f8431f252b21a/metadata/condition_mapping.json


<h3 style="text-align:center"> checking to see if cuda is installed </h3>

In [109]:
if torch.cuda.is_available():
    print("Cuda")
else:
    print("CPU")

Cuda


<h3 style="text-align:center"> Preprocessing dataset </h3>

In [110]:
print(type(dataset))

<class 'datasets.dataset_dict.DatasetDict'>


In [111]:
dataset = dataset["train"].to_pandas()

In [112]:
dataset.head()

,sample_id,participant,demographic,stimuli,response,condition_num,task_num,prompt,reasoning,study_id
0,0,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",5,7,0,You are a survey respondent with the following...,"Upon encountering the information about Jaime,...",9nphm
1,1,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",3,7,1,You are a survey respondent with the following...,"Upon reading the scenario about Jaime, the res...",9nphm
2,2,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",6,6,0,You are a survey respondent with the following...,"Upon reading the description of Jaime, this re...",9nphm
3,3,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",5,6,1,You are a survey respondent with the following...,As an individual in their late 60s with a some...,9nphm
4,4,2,"{'age': 79, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",6,4,0,You are a survey respondent with the following...,The respondent is likely to reflect on their o...,9nphm


<p> Breaking down the demographic column into its own dataframe</p>

In [113]:
demographicDataframe = pd.concat([
    dataset[["study_id"]].reset_index(drop=True),
    pd.DataFrame(dataset["demographic"].tolist())
], axis=1)

In [114]:
print(demographicDataframe)

        study_id   age                            education  \
0          9nphm  34.0  Post grad study/professional degree   
1          9nphm  34.0  Post grad study/professional degree   
2          9nphm  67.0                    Bachelor's degree   
3          9nphm  67.0                    Bachelor's degree   
4          9nphm  79.0  Post grad study/professional degree   
...          ...   ...                                  ...   
2901385    ef6my  17.0                                  NaN   
2901386    ef6my  17.0                                  NaN   
2901387    ef6my  17.0                                  NaN   
2901388    ef6my  17.0                                  NaN   
2901389    ef6my  17.0                                  NaN   

                        employment ethnicity  gender  household_size  \
0        Employed as paid employee       NaN  Female             4.0   
1        Employed as paid employee       NaN  Female             4.0   
2                          

<p> to solve the na values problem i decide to fill in values from the previous row </p>

In [115]:
demographicDataframe = demographicDataframe.ffill()

In [116]:
print(demographicDataframe.isna().sum())

study_id                 0
age                      0
education                0
employment               0
ethnicity            16407
gender                   0
household_size           0
housing_ownership        0
housing_type             0
ideology                 0
income                   0
internet_access          0
location                 0
marital_status           0
metro_status             0
party_id                 0
phone_service            0
dtype: int64


noticing ethnicity is full of NA's so will apply backwards fill to it

In [117]:
demographicDataframe["ethnicity"] = demographicDataframe["ethnicity"].bfill()

In [118]:
print(demographicDataframe.isna().sum())

study_id             0
age                  0
education            0
employment           0
ethnicity            0
gender               0
household_size       0
housing_ownership    0
housing_type         0
ideology             0
income               0
internet_access      0
location             0
marital_status       0
metro_status         0
party_id             0
phone_service        0
dtype: int64


In [119]:
demographicDataframe.head()

,study_id,age,education,employment,ethnicity,gender,household_size,housing_ownership,housing_type,ideology,income,internet_access,location,marital_status,metro_status,party_id,phone_service
0,9nphm,34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
1,9nphm,34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
2,9nphm,67.0,Bachelor's degree,Retired,White,Male,2.0,Owned or being bought by you or someone in you...,A one-family house attached to one or more houses,Somewhat Conservative,150-175K+,Internet Household,Iowa,Married,Metro Area,Don't Lean/Independent/None,Cellphone only
3,9nphm,67.0,Bachelor's degree,Retired,White,Male,2.0,Owned or being bought by you or someone in you...,A one-family house attached to one or more houses,Somewhat Conservative,150-175K+,Internet Household,Iowa,Married,Metro Area,Don't Lean/Independent/None,Cellphone only
4,9nphm,79.0,Post grad study/professional degree,Retired,White,Male,2.0,Owned or being bought by you or someone in you...,A one-family house detached from any other house,Moderate,40-49K,Internet Household,Indiana,Married,Metro Area,Strong Democrat,Cellphone only


merging demographic and dataset together

In [120]:
demographicDataframe.info()

<class 'pandas.DataFrame'>
RangeIndex: 2901390 entries, 0 to 2901389
Data columns (total 17 columns):
 #   Column             Dtype  
---  ------             -----  
 0   study_id           str    
 1   age                float64
 2   education          str    
 3   employment         str    
 4   ethnicity          str    
 5   gender             str    
 6   household_size     float64
 7   housing_ownership  str    
 8   housing_type       str    
 9   ideology           str    
 10  income             str    
 11  internet_access    str    
 12  location           str    
 13  marital_status     str    
 14  metro_status       str    
 15  party_id           str    
 16  phone_service      str    
dtypes: float64(2), str(15)
memory usage: 1.1 GB


In [121]:
dataset.info()

<class 'pandas.DataFrame'>
RangeIndex: 2901390 entries, 0 to 2901389
Data columns (total 10 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   sample_id      int64 
 1   participant    int64 
 2   demographic    object
 3   stimuli        str   
 4   response       int64 
 5   condition_num  int64 
 6   task_num       int64 
 7   prompt         str   
 8   reasoning      str   
 9   study_id       str   
dtypes: int64(5), object(1), str(4)
memory usage: 7.0+ GB


In [122]:
dataset = pd.merge(dataset, 
         demographicDataframe.drop_duplicates(subset='study_id'), 
         on='study_id')
dataset.head()

,sample_id,participant,demographic,stimuli,response,condition_num,task_num,prompt,reasoning,study_id,...,housing_ownership,housing_type,ideology,income,internet_access,location,marital_status,metro_status,party_id,phone_service
0,0,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",5,7,0,You are a survey respondent with the following...,"Upon encountering the information about Jaime,...",9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
1,1,0,"{'age': 34, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",3,7,1,You are a survey respondent with the following...,"Upon reading the scenario about Jaime, the res...",9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
2,2,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",6,6,0,You are a survey respondent with the following...,"Upon reading the description of Jaime, this re...",9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
3,3,1,"{'age': 67, 'education': 'Bachelor's degree', ...","You read ""Jaime is 20 years old and was born a...",5,6,1,You are a survey respondent with the following...,As an individual in their late 60s with a some...,9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
4,4,2,"{'age': 79, 'education': 'Post grad study/prof...","You read ""Jaime is 20 years old and was born a...",6,4,0,You are a survey respondent with the following...,The respondent is likely to reflect on their o...,9nphm,...,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only


dropping useless columns

In [124]:
dataset = dataset.drop("demographic", axis=1)
dataset = dataset.drop("sample_id", axis=1)
dataset = dataset.drop("participant", axis=1)
dataset = dataset.drop("condition_num", axis=1)
dataset = dataset.drop("task_num", axis=1)
dataset = dataset.drop("study_id", axis=1)


<h3 style="text-align:center"> Tokenisation/Fine tuning</h3>

In [126]:
dataset.head()

,stimuli,response,prompt,reasoning,age,education,employment,ethnicity,gender,household_size,housing_ownership,housing_type,ideology,income,internet_access,location,marital_status,metro_status,party_id,phone_service
0,"You read ""Jaime is 20 years old and was born a...",5,You are a survey respondent with the following...,"Upon encountering the information about Jaime,...",34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
1,"You read ""Jaime is 20 years old and was born a...",3,You are a survey respondent with the following...,"Upon reading the scenario about Jaime, the res...",34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
2,"You read ""Jaime is 20 years old and was born a...",6,You are a survey respondent with the following...,"Upon reading the description of Jaime, this re...",34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
3,"You read ""Jaime is 20 years old and was born a...",5,You are a survey respondent with the following...,As an individual in their late 60s with a some...,34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only
4,"You read ""Jaime is 20 years old and was born a...",6,You are a survey respondent with the following...,The respondent is likely to reflect on their o...,34.0,Post grad study/professional degree,Employed as paid employee,White,Female,4.0,Owned or being bought by you or someone in you...,A building with 2 or more apartments,Moderate,75-99K,Internet Household,New Jersey,Married,Metro Area,Moderate Democrat,Cellphone only


In [85]:
dataset["reasoning"].loc[0]

'Upon encountering the information about Jaime, a typical person in this persona may consider their own moderate stance towards gender identity issues and societal changes. They might reflect on the evolving nature of gender identity in contemporary society, recognizing that many individuals explore and change their identities over time. Given Jaime’s earlier feelings as a girl combined with the current non-binary identification, this respondent may weigh the likelihood of Jaime’s continued non-binary identification based on personal beliefs and societal acceptance. Additionally, the age factor plays a role in understanding that younger individuals may experience varied identity journeys. Overall, they may lean towards a belief that Jaime could still identify as non-binary in five years, but not without uncertainty.'

In [86]:
print(dataset["response"].min())
print(dataset["response"].max())

-5
60050000


splitting the dataset into 70% train, 20% evaluation and 10% testing

finetuning the model

In [47]:
model_name = "meta-llama/Meta-Llama-3.1-8B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_name) #loads in tokenizer associated with LLama 3.1-8B
model = AutoModelForCausalLM.from_pretrained(model_name)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    tokenizer=tokenizer,
)
trainer.train()

OSError: You are trying to access a gated repo.
Make sure to have access to it at https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct.
401 Client Error. (Request ID: Root=1-6a3d04af-34299dd42879583139021f8c;a8964785-aa0b-4670-976d-20b93684b5e0)

Cannot access gated repo for url https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct/resolve/main/config.json.
Access to model meta-llama/Llama-3.1-8B-Instruct is restricted. You must have access to it and be authenticated to access it. Please log in.